<div style="
    background: linear-gradient(135deg, #0f172a, #1d4ed8);
    padding: 28px;
    border-radius: 14px;
    color: white;
    text-align: center;
    margin-bottom: 25px;">
    <h1 style="color: white; margin: 0; font-size: 30px;">
        Analyse du Marché Immobilier à Tanger
    </h1>
    <p style="color: #dbeafe; font-size: 15px; margin-bottom: 0;">
        Analyse de données • Nettoyage des données • Analyse exploratoire • Modélisation
    </p>
</div>

## Contexte général

Ce projet a pour objectif d’analyser le marché immobilier de Tanger à partir d’un dataset contenant les caractéristiques des biens immobiliers, notamment leur type, localisation, superficie, nombre de pièces, état et ancienneté.

L’objectif principal est d’identifier les facteurs qui influencent les prix immobiliers et de développer un modèle de Machine Learning capable de prédire le prix de vente d’un bien à partir de ses caractéristiques.

Le projet comprend les étapes suivantes : **nettoyage des données (Data Cleaning), analyse exploratoire (EDA), visualisation et modélisation prédictive.**

### Description des colonnes du dataset immobilier de Tanger :

| Colonne | Description |
| :--- | :--- |
| **Type** | Type de bien immobilier (ex. Appartement, Villa, etc.) |
| **Localisation** | Quartier ou zone géographique du bien à Tanger |
| **Area** | Superficie totale du bien (en $m^2$) |
| **Rooms** | Nombre total de pièces |
| **Bedrooms** | Nombre de chambres à coucher |
| **Bathrooms** | Nombre de salles de bains |
| **Current_state** | État actuel ou condition du bien |
| **Age** | Âge du bien immobilier (en années) |
| **Price** | Prix de vente du bien (en MAD) |

<div style="
    background: linear-gradient(135deg, #0f172a, #1d4ed8);
    padding: 28px;
    border-radius: 14px;
    color: white;
    text-align: center;
    margin-bottom: 25px;">
    <h1 style="color: white; margin: 0; font-size: 30px;">
        1. Importation des bibliothèques
    </h1>
</div>

In [75]:
import math
import re

from statistics import linear_regression
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    MinMaxScaler,
    Normalizer,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

from catboost import CatBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier 
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    auc,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_curve,
)

import joblib
from sklearn.ensemble import IsolationForest


In [76]:
df = pd.read_csv("tanger_immobilier_dirty_pour_cleaning.csv")
df.head()

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberie,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0
4,Appartement,Mghogha,NaN,3.0,2.0,3.0,bon état,14.0,845000.0


---
### Résumé de la partie 1 : Importation

- Import des bibliothèques nécessaires : manipulation de données (`pandas`, `numpy`), prétraitement et pipelines (`scikit-learn`), visualisation (`plotly`, `seaborn`, `matplotlib`), modèles de régression (dont `XGBoost` et `CatBoost`), métriques d'évaluation et `joblib` pour la sauvegarde.
- Chargement du dataset brut `tanger_immobilier_dirty_pour_cleaning.csv` dans le DataFrame `df`, puis aperçu des premières lignes avec `df.head()`.

<div style="
    background: linear-gradient(135deg, #0f172a, #1d4ed8);
    padding: 28px;
    border-radius: 14px;
    color: white;
    text-align: center;
    margin-bottom: 25px;">
    <h1 style="color: white; margin: 0; font-size: 30px;">
        2. Compréhension des données
    </h1>
</div>

In [77]:
df.shape

(3035, 9)

In [78]:
df.columns

Index(['Type', 'Localisation', 'Area', 'Rooms', 'Bedrooms', 'Bathrooms',
       'Current_state', 'Age', 'Price'],
      dtype='str')

In [79]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3035 entries, 0 to 3034
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           2957 non-null   str    
 1   Localisation   2900 non-null   str    
 2   Area           2934 non-null   str    
 3   Rooms          2908 non-null   str    
 4   Bedrooms       2902 non-null   float64
 5   Bathrooms      2915 non-null   float64
 6   Current_state  2822 non-null   str    
 7   Age            2852 non-null   str    
 8   Price          2961 non-null   str    
dtypes: float64(2), str(7)
memory usage: 356.6 KB


### Vérification avant la conversion numérique

On repère les valeurs textuelles (ex. : `120 m²`, `1 200 000 DH`, `2,5M`, `quatre`) qui deviendraient `NaN` avec `errors='coerce'`. On les récupère avec la fonction `to_number` définie plus bas.

In [80]:
for col in ['Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Age', 'Price']:
    bad = df[pd.to_numeric(df[col], errors='coerce').isna() & df[col].notna()][col]
    print(col, '->', bad.unique()[:10])

Area -> <ArrowStringArray>
['env. 90', '1,200', '85m2', '120 m²', 'inconnu']
Length: 5, dtype: str
Rooms -> <ArrowStringArray>
['3 pièces', 'quatre']
Length: 2, dtype: str
Bedrooms -> []
Bathrooms -> []
Age -> <ArrowStringArray>
['10 ans', 'ancien']
Length: 2, dtype: str
Price -> <ArrowStringArray>
['2,5M', '1 200 000 DH', '—', '950000 MAD', 'inconnu']
Length: 5, dtype: str


In [81]:
mots = {'un': 1, 'deux': 2, 'trois': 3, 'quatre': 4, 'cinq': 5,
        'six': 6, 'sept': 7, 'huit': 8, 'neuf': 9, 'dix': 10}

def to_number(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()

    # nombres écrits en lettres : 'quatre' -> 4
    if val in mots:
        return mots[val]

    # '2,5M' -> 2 500 000
    if re.fullmatch(r'[\d.,]+\s*m', val):
        return float(val[:-1].replace(',', '.')) * 1000000

    # on enlève les unités o les espaces : '120 m²', '1 200 000 DH', '3 pièces', '10 ans', 'env. 90'
    val = re.sub(r'm²|m2|dh|mad|ans|pièces|pieces|env\.?|\s', '', val)
    val = val.replace(',', '')   # '1,200' -> 1200

    # 'inconnu', 'ancien', '—' ... -> NaN
    return pd.to_numeric(val, errors='coerce')

for col in ['Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Age', 'Price']:
    df[col] = df[col].apply(to_number)
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[['Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Age', 'Price']].isna().sum()

Area         103
Rooms        127
Bedrooms     133
Bathrooms    120
Age          186
Price         81
dtype: int64

In [82]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Area,2932.0,1.493591e+02,1.920822e+02,-15.0,73.0,97.0,139.25,2473.0
Rooms,2908.0,3.854195e+00,2.655950e+00,-2.0,2.0,3.0,5.00,29.0
Bedrooms,2902.0,2.712957e+00,2.109212e+00,-1.0,1.0,2.0,3.00,25.0
Bathrooms,2915.0,2.117667e+00,1.581100e+00,-1.0,1.0,2.0,3.00,20.0
Age,2849.0,1.826114e+01,1.475514e+01,-5.0,9.0,15.0,24.00,173.0
Price,2954.0,2.840695e+06,7.172903e+06,-1200000.0,1000000.0,1550000.0,2540000.00,114486532.0


### Interprétation générale de la table

`df.describe().T` permet de comparer, pour chaque variable, le minimum, les quartiles, la moyenne et le maximum. Trois points sont à surveiller :

- **Valeurs aberrantes (outliers) et illogiques** : un écart très important entre le maximum et le 75ᵉ percentile (superficie, prix) signale des valeurs extrêmes ou irréalistes.
- **Valeurs négatives inattendues** : un minimum négatif sur `Age`, `Price` ou le nombre de pièces est impossible dans la réalité et trahit une erreur de saisie.
- **Valeurs manquantes ou nulles** : un `count` inférieur au nombre total de lignes indique des données manquantes ; des valeurs égales à 0 (ex. prix) sont également suspectes.

Ces problèmes sont traités dans la partie suivante (nettoyage des données).

---
### Résumé de la partie 2 : Compréhension des données

- Exploration de la structure du dataset : dimensions (`df.shape`), colonnes (`df.columns`) et types de données (`df.info()`).
- Détection des valeurs textuelles dans les colonnes numériques (`Area`, `Rooms`, `Bedrooms`, `Bathrooms`, `Age`, `Price`).
- Conversion de ces colonnes en nombres avec la fonction `to_number` : nombres en lettres (`quatre` → 4), notation `2,5M` → 2 500 000, suppression des unités (`m²`, `DH`, `ans`, `pièces`…) ; les valeurs inexploitables deviennent `NaN`.
- Statistiques descriptives (`df.describe().T`) pour repérer les valeurs aberrantes, négatives ou manquantes.

<div style="
    background: linear-gradient(135deg, #0f172a, #1d4ed8);
    padding: 28px;
    border-radius: 14px;
    color: white;
    text-align: center;
    margin-bottom: 25px;">
    <h1 style="color: white; margin: 0; font-size: 30px;">
        3. Nettoyage des données
    </h1>
</div>

## Traitement des variables catégorielles

### Traitement de la colonne `Type`

In [83]:
df['Type'].value_counts()

Type
Appartement      1722
Maison            366
Villa             331
 appartement       86
Appart             85
APPARTEMENT        83
Apartement         80
appartement        70
MAISON             21
maison             20
Maisón             19
villa              18
VILLA              17
Vila               16
 maison            14
 villa              9
Name: count, dtype: int64

In [84]:
def clean_type(val):
    if pd.isna(val):
        return val
    val = str(val).strip().lower()
    
    if 'appart' in val or 'apart' in val:
        return 'Appartement'
    elif 'maison' in val:
        return 'Maison'
    elif 'villa' in val:
        return 'Villa'
    else:
        return val.capitalize()

# Appliquer la fonction
df['Type'] = df['Type'].apply(clean_type)

In [85]:
df['Type'] = df['Type'].replace({
    'Maisón': 'Maison',
    'Vila': 'Villa'
})

In [86]:
df['Type'].value_counts()

Type
Appartement    2126
Maison          440
Villa           391
Name: count, dtype: int64

In [87]:
df.head()

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberie,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0
4,Appartement,Mghogha,NaN,3.0,2.0,3.0,bon état,14.0,845000.0


### Traitement de la colonne `Localisation`

In [88]:
df['Localisation'].unique()

<ArrowStringArray>
[      'Centre Ville',             'Iberie',             'Centre',
          'Boukhalef',            'Mghogha',           'Branes 1',
       'Centre-Ville',           'Malabata',         'Aharrarine',
         'Boukhalef ',          'Gourziana',        'Tanja Balia',
           'Marchane',         'Nouinouich',        'Moujahidine',
           'MALABATA',        'De la Plage',      'Moulay Ismail',
             'Ahlane',          'Jbel Kbir',        'Beni Makada',
 'Tanger City Center',            'mesnana',           'Mediouna',
         ' Malabata ',            'Mesnana',          'BOUKHALEF',
            'MESNANA',            'marjane',              'Sania',
            'Marjane',             'Rmilat',            'Achakar',
      'Centre ville ',                  nan,          'Malabatta',
              'M'nar',        'tanja balia',         'Val Fleuri',
         'Casabarata',            'mghogha',       'Tanja Balia ',
           'malabata',     'Aouama Gharbia'

In [89]:
# 1. Dictionnaire standard pour regrouper les variantes par catégorie
standards = {
    # Centre Ville
    'centre ville': 'Centre Ville', 'centre-ville': 'Centre Ville', 
    'centre': 'Centre Ville', 'ctr ville': 'Centre Ville',
    
    # Iberia
    'iberie': 'Iberia', 'ibérie': 'Iberia', 'iberia': 'Iberia',
    
    # Mghogha
    'mghogha': 'Mghogha', 'mghougha': 'Mghogha',
    
    # Malabata
    'malabata': 'Malabata', 'malabatta': 'Malabata',
    
    # Tanja Balia
    'tanja balia': 'Tanja Balia', 'tanja-balia': 'Tanja Balia',
    
    # Marchane
    'marchane': 'Marchane', 'marchan': 'Marchane',
    
    # Jbel Kbir
    'jbel kbir': 'Jbel Kbir', 'jbel kebir': 'Jbel Kbir',
    
    # Casabarata
    'casabarata': 'Casabarata', 'casa barata': 'Casabarata',
    
    # Achakar
    'achakar': 'Achakar', 'achakkar': 'Achakar',
    
    # Val Fleuri
    'val fleuri': 'Val Fleuri', 'val-fleuri': 'Val Fleuri'
}

# 2. Nettoyage initial (espaces zaydin o minuscules)
clean = df['Localisation'].astype(str).str.strip().str.lower()

# 3. Application du standard (ila makanch f l-liste, kayakhod l'écriture propre b Title Case)
df['Localisation'] = clean.map(standards).fillna(clean.str.title())

# 4. Re-mettre les valeurs manquantes (NaN) en place
df.loc[df['Localisation'] == 'Nan', 'Localisation'] = None

In [90]:
df['Localisation'].unique()

<ArrowStringArray>
[      'Centre Ville',             'Iberia',          'Boukhalef',
            'Mghogha',           'Branes 1',           'Malabata',
         'Aharrarine',          'Gourziana',        'Tanja Balia',
           'Marchane',         'Nouinouich',        'Moujahidine',
        'De La Plage',      'Moulay Ismail',             'Ahlane',
          'Jbel Kbir',        'Beni Makada', 'Tanger City Center',
            'Mesnana',           'Mediouna',            'Marjane',
              'Sania',             'Rmilat',            'Achakar',
                  nan,              'M'Nar',         'Val Fleuri',
         'Casabarata',     'Aouama Gharbia',         'Ben Dibane']
Length: 30, dtype: str

In [91]:
df.head()

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberia,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre Ville,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0
4,Appartement,Mghogha,NaN,3.0,2.0,3.0,bon état,14.0,845000.0


In [92]:
df['Current_state'].unique()

<ArrowStringArray>
[             'Bon état',         'Très bon état',             'bon état ',
      'Bon état général',            'État moyen',             'A rénover',
             'À rénover', 'Renovation nécessaire',             'à rénover',
                     nan,              'BON ÉTAT',              'bon etat',
             'A renover',              'Bon Etat',                  'Neuf',
            'À rénover ',                  'NEUF',               'Nouveau',
                ' neuf ',                  'neuf',                 'Neuve']
Length: 21, dtype: str

In [93]:
# 1. Dictionnaire d les catégories standard
standards_etat = {
    # Neuf
    'neuf': 'Neuf', 'neuve': 'Neuf', 'nouveau': 'Neuf',
    
    # Bon état
    'bon etat': 'Bon état', 'bon état': 'Bon état', 'bon etat general': 'Bon état', 'bon état général': 'Bon état',
    
    # Très bon état
    'tres bon etat': 'Très bon état', 'très bon état': 'Très bon état',
    
    # État moyen
    'etat moyen': 'État moyen', 'état moyen': 'État moyen',
    
    # À rénover
    'a renover': 'À rénover', 'à rénover': 'À rénover', 'a rénover': 'À rénover',
    'renovation necessaire': 'À rénover', 'rénovation nécessaire': 'À rénover', 'renovation nécessaire': 'À rénover'
}

# 2. Nettoyage d les espaces o les minuscules
clean_etat = df['Current_state'].astype(str).str.strip().str.lower()

# Vérification : valeurs li ma kaynach f dictionnaire (khass ytb9a ghir 'nan')
print(clean_etat[~clean_etat.isin(standards_etat)].unique())

# 3. Application dyal l-mapping
df['Current_state'] = clean_etat.map(standards_etat)

# 4. Ila kanet chi valeur nan / missing
df.loc[clean_etat == 'nan', 'Current_state'] = None

<ArrowStringArray>
[nan]
Length: 1, dtype: str


In [94]:
df['Current_state'].value_counts()

Current_state
Bon état         1228
État moyen        716
À rénover         445
Très bon état     395
Neuf               38
Name: count, dtype: int64

In [95]:
df.head()

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberia,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre Ville,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0
4,Appartement,Mghogha,NaN,3.0,2.0,3.0,Bon état,14.0,845000.0


## Traitement des variables numériques

### Traitement de la colonne `Area`

In [96]:
df['Area'].mean()

np.float64(149.35914051841746)

In [97]:
df['Area'].dtypes

dtype('float64')

In [98]:
df['Area'].describe()

count    2932.000000
mean      149.359141
std       192.082181
min       -15.000000
25%        73.000000
50%        97.000000
75%       139.250000
max      2473.000000
Name: Area, dtype: float64

In [99]:
fig = px.box(
    df,
    x= df['Area'],
    title='Area boxplot avant nettoyage'
)
fig.update_xaxes(title_text='Area (m²)')
fig.update_yaxes(title_text='Count')
fig.show()

In [100]:
# Valeurs aberrantes -> NaN (ma kan7ydouch la ligne, ghadi n3awdo b la médiane mn be3d)
df.loc[(df['Area'] < 50) | (df['Area'] > 1500), 'Area'] = np.nan

fig = px.box(df, x='Area', title='Area boxplot après nettoyage')
fig.show()

### Traitement de la colonne `Rooms`

In [101]:
df.head(1)

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0


In [102]:
df['Rooms'].value_counts()

Rooms
 3.0     766
 2.0     624
 4.0     484
 5.0     276
 1.0     260
 6.0     165
 7.0      90
 8.0      63
 9.0      53
 10.0     35
 11.0     25
 12.0     19
 16.0      9
 13.0      6
 14.0      6
 0.0       5
-2.0       5
 15.0      4
 17.0      3
 18.0      2
 23.0      2
 22.0      1
 21.0      1
 28.0      1
 19.0      1
 29.0      1
 24.0      1
Name: count, dtype: int64

In [103]:
df['Rooms'].dtype

dtype('float64')

In [104]:
df['Rooms'].describe()

count    2908.000000
mean        3.854195
std         2.655950
min        -2.000000
25%         2.000000
50%         3.000000
75%         5.000000
max        29.000000
Name: Rooms, dtype: float64

In [105]:
fig = px.box(
    df, 
    x='Rooms',
    title='Rooms boxplot',
)

fig.show()

In [106]:
df.loc[(df['Rooms'] < 1) | (df['Rooms'] > 10), 'Rooms'] = np.nan
fig = px.box(
    df,
    x='Rooms',
    title='Rooms boxplot après nettoyage'
)
fig.show()

### Traitement de la colonne `Bedrooms`

In [107]:
print(df['Bedrooms'].describe())
print(df['Bedrooms'].value_counts().sort_index())

count    2902.000000
mean        2.712957
std         2.109212
min        -1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        25.000000
Name: Bedrooms, dtype: float64
Bedrooms
-1.0       4
 0.0     213
 1.0     546
 2.0     801
 3.0     615
 4.0     337
 5.0     164
 6.0     103
 7.0      43
 8.0      31
 9.0      18
 10.0      9
 11.0      3
 12.0      4
 13.0      1
 14.0      1
 16.0      1
 19.0      1
 20.0      1
 21.0      3
 23.0      2
 25.0      1
Name: count, dtype: int64


In [108]:
df.loc[(df['Bedrooms'] < 1) | (df['Bedrooms'] > 10), 'Bedrooms'] = np.nan
fig = px.box(
    df,
    x='Bedrooms',
    title='Bedrooms boxplot après nettoyage'
)
fig.show()

### Traitement de la colonne `Bathrooms`

In [109]:
df['Bathrooms'].describe()

count    2915.000000
mean        2.117667
std         1.581100
min        -1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max        20.000000
Name: Bathrooms, dtype: float64

In [110]:
df.loc[(df['Bathrooms'] < 1) | (df['Bathrooms'] > 10), 'Bathrooms'] = np.nan
fig = px.box(
    df,
    x='Bathrooms',
    title='Bathrooms boxplot après nettoyage'
)
fig.show()

### Traitement de la colonne `Age`

In [111]:
df.head()

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberia,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre Ville,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0
4,Appartement,Mghogha,NaN,3.0,2.0,3.0,Bon état,14.0,845000.0


In [112]:
df['Age'].describe()

count    2849.000000
mean       18.261144
std        14.755138
min        -5.000000
25%         9.000000
50%        15.000000
75%        24.000000
max       173.000000
Name: Age, dtype: float64

In [113]:
fig = px.box(
    df,
    x='Age',
    title='Age boxplot avant nettoyage'
)
fig.show()

In [114]:
df.loc[(df['Age'] < 0) | (df['Age'] > 70), 'Age'] = np.nan
fig = px.box(
    df,
    x='Age',
    title='Age boxplot après nettoyage'
)
fig.show()

### Traitement de la colonne `Price` (variable cible)

In [115]:
fig = px.box(
    df,
    x='Price',
    title='Price boxplot avant nettoyage'
)
fig.show()

In [116]:
# Prix <= 0 -> NaN (l'imputation kat3awd mn be3d f la partie valeurs nulls)
df.loc[df['Price'] <= 0, 'Price'] = np.nan

In [117]:
fig = px.box(
    df,
    x='Price',
    title='Price boxplot avant nettoyage-1'
)
fig.show()

In [118]:
# Prix trop petits ou trop grands (outliers) -> NaN
df.loc[(df['Price'] < 100000) | (df['Price'] > 15000000), 'Price'] = np.nan

In [119]:
fig = px.box(
    df,
    x='Price',
    title='Price boxplot apres nettoyage-2'
)
fig.show()

In [120]:
df.isnull().sum()

Type              78
Localisation     135
Area             277
Rooms            219
Bedrooms         368
Bathrooms        143
Current_state    213
Age              209
Price            123
dtype: int64

### Doublons et cohérence (avant l'imputation)

In [121]:
# Doublons (9bl l'imputation, bach l'imputation ma t3tich des lignes identiques)
print("Doublons :", df.duplicated().sum())
df = df.drop_duplicates()

# Bedrooms > Rooms (les 2 valeurs kaynin) = ligne incohérente
print("Bedrooms > Rooms (avant imputation) :", (df['Bedrooms'] > df['Rooms']).sum())
df = df[~(df['Bedrooms'] > df['Rooms'])].copy()

df.shape

Doublons : 35
Bedrooms > Rooms (avant imputation) : 4


(2996, 9)

## Traitement des valeurs manquantes

### Variables catégorielles (imputation par le mode)

In [122]:
fig = px.pie(
    df,
    names='Type',
    title='Répartition des types de biens',
    hole=0.4
)

fig.show()

In [123]:
fig = px.pie(
    df,
    names='Localisation',
    title='Répartition des Localisation de biens',
    hole=0.4
)

fig.show()

In [124]:
fig = px.pie(
    df,
    names='Current_state',
    title='Répartition des current_state de biens',
   
)

fig.show()

In [125]:
df['Type'] = df['Type'].fillna(df['Type'].mode()[0])                  
df['Current_state'] = df['Current_state'].fillna(df['Current_state'].mode()[0])
df['Localisation'] = df['Localisation'].fillna(df['Localisation'].mode()[0])    

fig = px.pie(
    df, 
    names='Type', 
    title='Répartition par Type de bien (Sans Unknown)',
    hole=0.45
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show() 

fig = px.pie(
    df,
    names='Current_state',
    title='Répartition des current_state de biens',
   
)

fig.show()

fig = px.pie(
    df,
    names='Localisation',
    title='Répartition des Localisation de biens',
    hole=0.4
)

fig.show()

In [126]:
df.isnull().sum()

Type               0
Localisation       0
Area             274
Rooms            216
Bedrooms         364
Bathrooms        140
Current_state      0
Age              207
Price            123
dtype: int64

### Variables numériques (imputation)

In [127]:
# Area, Rooms, Bedrooms, Bathrooms, Age -> médiane par Type
for col in ['Area', 'Rooms', 'Bedrooms', 'Bathrooms', 'Age']:
    df[col] = df[col].fillna(df.groupby('Type')[col].transform('median'))

# Rooms, Bedrooms, Bathrooms khasshom ykono des entiers
for col in ['Rooms', 'Bedrooms', 'Bathrooms']:
    df[col] = df[col].round()

# Price (target) : ma kan3awdoch b la médiane (kaydir des faux prix), on supprime les lignes
df = df.dropna(subset=['Price'])

# Après imputation, Rooms khass ykon >= Bedrooms
print("Bedrooms > Rooms (après imputation) :", (df['Bedrooms'] > df['Rooms']).sum())
df['Rooms'] = df[['Rooms', 'Bedrooms']].max(axis=1)

Bedrooms > Rooms (après imputation) : 268


In [128]:
df.isnull().sum()

Type             0
Localisation     0
Area             0
Rooms            0
Bedrooms         0
Bathrooms        0
Current_state    0
Age              0
Price            0
dtype: int64

## Vérification finale

In [129]:
df = df.reset_index(drop=True)
df.shape

(2873, 9)

### Contrôle de la structure et des statistiques

Dernière vérification du dataset nettoyé, puis sauvegarde dans `tanger_immobilier_cleaned.csv`.

In [130]:
df.describe()

,Area,Rooms,Bedrooms,Bathrooms,Age,Price
count,2873.000000,2873.000000,2873.000000,2873.000000,2873.000000,2.873000e+03
mean,140.085625,3.736512,2.745562,2.037939,17.507135,2.243404e+06
std,131.143759,1.887647,1.536579,1.112862,11.303827,2.140740e+06
min,50.000000,1.000000,1.000000,1.000000,0.000000,2.550000e+05
25%,79.000000,2.000000,2.000000,1.000000,9.000000,1.000000e+06
50%,97.000000,3.000000,2.000000,2.000000,15.000000,1.530000e+06
75%,136.000000,5.000000,3.000000,3.000000,23.000000,2.480000e+06
max,1483.000000,10.000000,10.000000,8.000000,65.000000,1.499500e+07


In [131]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2873 entries, 0 to 2872
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           2873 non-null   str    
 1   Localisation   2873 non-null   str    
 2   Area           2873 non-null   float64
 3   Rooms          2873 non-null   float64
 4   Bedrooms       2873 non-null   float64
 5   Bathrooms      2873 non-null   float64
 6   Current_state  2873 non-null   str    
 7   Age            2873 non-null   float64
 8   Price          2873 non-null   float64
dtypes: float64(6), str(3)
memory usage: 286.2 KB


In [132]:
df.head(10)

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberia,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre Ville,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0
4,Appartement,Mghogha,91.0,3.0,2.0,3.0,Bon état,14.0,845000.0
5,Appartement,Branes 1,94.0,3.0,3.0,3.0,Bon état,19.0,1450000.0
6,Maison,Centre Ville,253.0,6.0,6.0,4.0,État moyen,30.0,5380000.0
7,Appartement,Malabata,96.0,4.0,3.0,2.0,Bon état,4.0,2235000.0
8,Appartement,Malabata,53.0,3.0,2.0,2.0,Très bon état,7.0,1235000.0
9,Appartement,Aharrarine,175.0,6.0,5.0,4.0,Très bon état,8.0,1560000.0


In [133]:
df.to_csv("tanger_immobilier_cleaned.csv", index=False)

---
### Résumé de la partie 3 : Nettoyage des données

**Variables catégorielles**

| Colonne | Traitement |
| :--- | :--- |
| `Type` | Normalisation de la casse et des variantes (`appart`, `maison`, `villa`), puis correction des fautes (`Maisón` → `Maison`, `Vila` → `Villa`). |
| `Localisation` | Regroupement des variantes d'écriture (ex. `iberie`/`ibérie` → `Iberia`, `centre-ville` → `Centre Ville`) via un dictionnaire de standardisation. |
| `Current_state` | Standardisation des états (`Neuf`, `Très bon état`, `Bon état`, `État moyen`, `À rénover`). |

**Variables numériques** : les valeurs hors seuils sont remplacées par `NaN` après analyse des boxplots.

| Colonne | Règle appliquée |
| :--- | :--- |
| `Area` | Valeurs < 50 ou > 1500 m² |
| `Rooms`, `Bedrooms`, `Bathrooms` | Valeurs < 1 ou > 10 |
| `Age` | Valeurs < 0 ou > 70 ans |
| `Price` | Prix ≤ 0, puis < 100 000 ou > 15 000 000 MAD |

**Cohérence** : suppression des doublons et des lignes où `Bedrooms` > `Rooms`.

**Valeurs manquantes**

- Variables catégorielles : imputation par le **mode**.
- Variables numériques : imputation par la **médiane selon le `Type`** ; arrondi des nombres de pièces, chambres et salles de bains.
- `Price` (cible) : les lignes sans prix sont **supprimées** (pas d'imputation pour ne pas fabriquer de faux prix).
- Contrôle final de cohérence (`Rooms` ≥ `Bedrooms`), réinitialisation de l'index et sauvegarde dans `tanger_immobilier_cleaned.csv`.

<div style="
    background: linear-gradient(135deg, #0f172a, #1d4ed8);
    padding: 28px;
    border-radius: 14px;
    color: white;
    text-align: center;
    margin-bottom: 25px;">
    <h1 style="color: white; margin: 0; font-size: 30px;">
        4. Prétraitement
    </h1>
</div>

In [134]:
df.head(4)

,Type,Localisation,Area,Rooms,Bedrooms,Bathrooms,Current_state,Age,Price
0,Appartement,Centre Ville,78.0,3.0,2.0,2.0,Bon état,6.0,1300000.0
1,Appartement,Iberia,122.0,3.0,2.0,2.0,Bon état,13.0,3050000.0
2,Appartement,Centre Ville,81.0,3.0,2.0,1.0,Très bon état,5.0,1335000.0
3,Appartement,Boukhalef,129.0,5.0,4.0,3.0,Bon état,28.0,1315000.0


In [135]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2873 entries, 0 to 2872
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           2873 non-null   str    
 1   Localisation   2873 non-null   str    
 2   Area           2873 non-null   float64
 3   Rooms          2873 non-null   float64
 4   Bedrooms       2873 non-null   float64
 5   Bathrooms      2873 non-null   float64
 6   Current_state  2873 non-null   str    
 7   Age            2873 non-null   float64
 8   Price          2873 non-null   float64
dtypes: float64(6), str(3)
memory usage: 286.2 KB


## Transformation des variables numériques

### Transformation logarithmique (`log1p`) de `Area` et `Price`

Cette transformation réduit l'asymétrie des distributions (prix et superficie très étalés vers les grandes valeurs). Les histogrammes avant/après permettent de le vérifier.

In [136]:
df['Area_log'] = np.log1p(df['Area'])
df['Price_log'] = np.log1p(df['Price'])

In [137]:
fig = px.histogram(df, x='Price', nbins=50, title='Price avant log')
fig.show()

fig = px.histogram(df, x='Price_log', nbins=50, title='Price après log1p')
fig.show()

## Séparation des colonnes (numériques / nominales / ordinales)

In [138]:
cols_num = ['Area_log', 'Rooms', 'Bedrooms', 'Bathrooms', 'Age']   # numériques
cols_nominal = ['Type', 'Localisation']                           # pas d'ordre -> OneHot
cols_ordinal = ['Current_state']                                  # kayn ordre -> Ordinal

# l'ordre de l'état : de moins bon l mzyan
ordre_etat = [['À rénover', 'État moyen', 'Bon état', 'Très bon état', 'Neuf']]

X = df[cols_num + cols_nominal + cols_ordinal]
y = df['Price_log']

print(X.shape, y.shape)
X.head()

(2873, 8) (2873,)


,Area_log,Rooms,Bedrooms,Bathrooms,Age,Type,Localisation,Current_state
0,4.369448,3.0,2.0,2.0,6.0,Appartement,Centre Ville,Bon état
1,4.812184,3.0,2.0,2.0,13.0,Appartement,Iberia,Bon état
2,4.406719,3.0,2.0,1.0,5.0,Appartement,Centre Ville,Très bon état
3,4.867534,5.0,4.0,3.0,28.0,Appartement,Boukhalef,Bon état
4,4.521789,3.0,2.0,3.0,14.0,Appartement,Mghogha,Bon état


## Découpage des données : train (70 %) / validation (15 %) / test (15 %)

In [139]:
# 1. 70% train / 30% reste
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

# 2. 30% li b9aw -> 15% validation / 15% test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print("Train :", X_train.shape)
print("Validation :", X_val.shape)
print("Test :", X_test.shape)

Train : (2011, 8)
Validation : (431, 8)
Test : (431, 8)


## Pipeline de prétraitement

Le `fit` est effectué **uniquement sur le train** afin d'éviter toute fuite de données (*data leakage*). La même transformation est ensuite appliquée à la validation et au test.

In [140]:
preprocessor = ColumnTransformer(
    transformers=[
        # numériques -> StandardScaler
        ('num', StandardScaler(), cols_num),

        # nominales -> OneHotEncoder (handle_unknown: ila jat localisation jdida f test)
        ('nominal', OneHotEncoder(handle_unknown='ignore'), cols_nominal),

        # ordinale -> OrdinalEncoder + StandardScaler (bach ykon 3ndha nfs poids f KNN / SVR)
        ('ordinal', Pipeline([
            ('encoder', OrdinalEncoder(categories=ordre_etat)),
            ('scaler', StandardScaler())
        ]), cols_ordinal)
    ],
    sparse_threshold=0   # matrice normale (dense) machi sparse
)

In [141]:
# vérification : chhal men colonne ba9a mn be3d l'encodage
X_train_prep = preprocessor.fit_transform(X_train)
print(X_train_prep.shape)
print(preprocessor.get_feature_names_out())

(2011, 38)
['num__Area_log' 'num__Rooms' 'num__Bedrooms' 'num__Bathrooms' 'num__Age'
 'nominal__Type_Appartement' 'nominal__Type_Maison' 'nominal__Type_Villa'
 'nominal__Localisation_Achakar' 'nominal__Localisation_Aharrarine'
 'nominal__Localisation_Ahlane' 'nominal__Localisation_Aouama Gharbia'
 'nominal__Localisation_Ben Dibane' 'nominal__Localisation_Beni Makada'
 'nominal__Localisation_Boukhalef' 'nominal__Localisation_Branes 1'
 'nominal__Localisation_Casabarata' 'nominal__Localisation_Centre Ville'
 'nominal__Localisation_De La Plage' 'nominal__Localisation_Gourziana'
 'nominal__Localisation_Iberia' 'nominal__Localisation_Jbel Kbir'
 "nominal__Localisation_M'Nar" 'nominal__Localisation_Malabata'
 'nominal__Localisation_Marchane' 'nominal__Localisation_Marjane'
 'nominal__Localisation_Mediouna' 'nominal__Localisation_Mesnana'
 'nominal__Localisation_Mghogha' 'nominal__Localisation_Moujahidine'
 'nominal__Localisation_Moulay Ismail' 'nominal__Localisation_Nouinouich'
 'nominal__Lo

---
### Résumé de la partie 4 : Prétraitement

- **Transformation logarithmique** (`log1p`) de `Area` et `Price` (création de `Area_log` et `Price_log`), avec comparaison des histogrammes avant/après.
- **Séparation des variables** :
  - numériques : `Area_log`, `Rooms`, `Bedrooms`, `Bathrooms`, `Age` ;
  - nominales : `Type`, `Localisation` ;
  - ordinale : `Current_state` (ordre : À rénover < État moyen < Bon état < Très bon état < Neuf).
- **Variable cible** : `Price_log`.
- **Découpage** des données : 70 % train, 15 % validation, 15 % test (`random_state=42`).
- **Pipeline de prétraitement** (`ColumnTransformer`) : `StandardScaler` pour les numériques, `OneHotEncoder` pour les nominales, `OrdinalEncoder` + `StandardScaler` pour l'ordinale. Le `fit` se fait uniquement sur le train pour éviter le data leakage.

<div style="
    background: linear-gradient(135deg, #0f172a, #1d4ed8);
    padding: 28px;
    border-radius: 14px;
    color: white;
    text-align: center;
    margin-bottom: 25px;">
    <h1 style="color: white; margin: 0; font-size: 30px;">
        5. Modélisation
    </h1>
</div>

## 1. Les modèles de régression

Tous les modèles sont regroupés dans un dictionnaire (nom : modèle). Pour en ajouter ou en retirer un, il suffit de modifier ce dictionnaire.

In [142]:
from sklearn.base import clone

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'SVR': SVR(C=10),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42),
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.05, verbose=0, random_state=42)
}

## 2. Fonction d'évaluation de tous les modèles

Chaque modèle est entraîné dans un **Pipeline** (prétraitement + modèle). La fonction retourne :

- `tableau` : R² train / validation / test et les erreurs ;
- `pipes` : les modèles entraînés ;
- `meilleur` : le nom du meilleur modèle (selon `critere`).

Le R² est calculé sur `Price_log` (l'échelle sur laquelle les modèles sont entraînés), tandis que les erreurs `MAE` et `RMSE` sont exprimées en **MAD** (prix réel, après `expm1`).

In [143]:
def evaluer_modeles(models, preprocessor, X_train, y_train, X_val, y_val, X_test, y_test, critere='R2_val'):
    resultats = []
    pipes = {}

    # les vrais prix (MAD) bach n7sbo les erreurs
    reel_val = np.expm1(y_val)
    reel_test = np.expm1(y_test)

    for nom, model in models.items():
        # preprocessing + model f pipeline wa7da
        pipe = Pipeline([
            ('prep', clone(preprocessor)),
            ('model', clone(model))
        ])
        pipe.fit(X_train, y_train)
        pipes[nom] = pipe

        # prédictions (échelle log)
        pred_train = pipe.predict(X_train)
        pred_val = pipe.predict(X_val)
        pred_test = pipe.predict(X_test)

        r2_train = r2_score(y_train, pred_train)
        r2_val = r2_score(y_val, pred_val)
        r2_test = r2_score(y_test, pred_test)

        resultats.append({
            'Modèle': nom,
            'R2_train': r2_train,
            'R2_val': r2_val,
            'R2_test': r2_test,
            'Overfit': r2_train - r2_test,
            'MAE_val': mean_absolute_error(reel_val, np.expm1(pred_val)),
            'MAE_test': mean_absolute_error(reel_test, np.expm1(pred_test)),
            'RMSE_val': np.sqrt(mean_squared_error(reel_val, np.expm1(pred_val))),
            'RMSE_test': np.sqrt(mean_squared_error(reel_test, np.expm1(pred_test)))
        })

    # trier : R2 -> l akbar fo9, erreurs -> l asghar fo9
    croissant = critere not in ['R2_train', 'R2_val', 'R2_test']
    tableau = pd.DataFrame(resultats).sort_values(critere, ascending=croissant).reset_index(drop=True)
    meilleur = tableau.loc[0, 'Modèle']

    return tableau, pipes, meilleur

## 3. Entraînement et tableau de comparaison

In [144]:
tableau, pipes, meilleur = evaluer_modeles(
    models, preprocessor,
    X_train, y_train, X_val, y_val, X_test, y_test,
    critere='R2_val'
)

tableau.style.format({
    'R2_train': '{:.3f}', 'R2_val': '{:.3f}', 'R2_test': '{:.3f}', 'Overfit': '{:.3f}',
    'MAE_val': '{:,.0f}', 'MAE_test': '{:,.0f}', 'RMSE_val': '{:,.0f}', 'RMSE_test': '{:,.0f}'
}).highlight_max(subset=['R2_val', 'R2_test'], color='lightgreen')

,Modèle,R2_train,R2_val,R2_test,Overfit,MAE_val,MAE_test,RMSE_val,RMSE_test
0,CatBoost,0.939,0.884,0.917,0.022,"409,443","378,448","741,627","701,184"
1,XGBoost,0.959,0.872,0.907,0.052,"418,914","382,234","755,909","698,827"
2,Gradient Boosting,0.894,0.862,0.893,0.002,"443,682","392,651","767,889","690,919"
3,Random Forest,0.982,0.856,0.887,0.095,"448,732","402,892","740,307","693,326"
4,SVR,0.948,0.842,0.848,0.100,"444,307","481,014","766,597","885,736"
5,Lasso,0.832,0.821,0.848,-0.016,"539,196","520,515","923,823","903,648"
6,Ridge,0.834,0.821,0.849,-0.015,"543,907","528,308","945,995","929,819"
7,Linear Regression,0.834,0.821,0.849,-0.015,"545,525","530,109","951,242","935,886"
8,KNN,0.870,0.798,0.812,0.058,"533,079","527,517","847,054","844,793"
9,Decision Tree,0.823,0.745,0.778,0.045,"567,523","553,079","974,043","917,204"


## 4. Le meilleur modèle

Le meilleur modèle est choisi selon le **R² de validation** (`critere='R2_val'`). Le **R² de test** reste le score final réellement fiable, car le jeu de test n'intervient pas dans le choix. Pour choisir selon le test, remplacer par `critere='R2_test'`.

Le pipeline complet (prétraitement + modèle) est sauvegardé avec `joblib`.

In [145]:
ligne = tableau.iloc[0]

print(f"Meilleur modèle : {meilleur}")
print(f"R2 train = {ligne['R2_train']:.3f} | R2 validation = {ligne['R2_val']:.3f} | R2 test = {ligne['R2_test']:.3f}")
print(f"MAE test = {ligne['MAE_test']:,.0f} MAD | RMSE test = {ligne['RMSE_test']:,.0f} MAD")

# sauvegarde dyal l pipeline kamel (preprocessing + model)
joblib.dump(pipes[meilleur], 'meilleur_modele.pkl')

Meilleur modèle : CatBoost
R2 train = 0.939 | R2 validation = 0.884 | R2 test = 0.917
MAE test = 378,448 MAD | RMSE test = 701,184 MAD


['meilleur_modele.pkl']

## 5. Graphiques de comparaison (train / validation / test et erreurs)

In [146]:
def graphes_comparaison(tableau):
    # 1. R2 : train vs validation vs test
    r2 = tableau.melt(id_vars='Modèle', value_vars=['R2_train', 'R2_val', 'R2_test'],
                      var_name='Jeu', value_name='R2')
    fig = px.bar(r2, x='Modèle', y='R2', color='Jeu', barmode='group',
                 title='R2 : train vs validation vs test')
    fig.show()

    # 2. overfitting : écart R2 train - R2 test
    fig = px.bar(tableau, x='Modèle', y='Overfit',
                 title='Overfitting : R2 train - R2 test (kol ma kber, kol ma overfit)')
    fig.show()

    # 3. erreurs en MAD : validation vs test
    for metrique in ['MAE', 'RMSE']:
        err = tableau.melt(id_vars='Modèle', value_vars=[f'{metrique}_val', f'{metrique}_test'],
                           var_name='Jeu', value_name='Erreur (MAD)')
        fig = px.bar(err, x='Modèle', y='Erreur (MAD)', color='Jeu', barmode='group',
                     title=f'{metrique} : validation vs test')
        fig.show()

graphes_comparaison(tableau)

## 6. Graphiques du meilleur modèle

In [147]:
def graphes_meilleur_modele(pipe, nom, X_test, y_test):
    reel = np.expm1(y_test)
    pred = np.expm1(pipe.predict(X_test))
    residus = reel - pred

    # 1. prix réel vs prix prédit (kol ma 9rb mn l khat, mzyan)
    fig = px.scatter(x=reel, y=pred, opacity=0.6,
                     labels={'x': 'Prix réel (MAD)', 'y': 'Prix prédit (MAD)'},
                     title=f'{nom} : prix réel vs prix prédit (test)')
    fig.add_shape(type='line', x0=reel.min(), y0=reel.min(), x1=reel.max(), y1=reel.max(),
                  line=dict(color='red', dash='dash'))
    fig.show()

    # 2. erreurs (résidus) vs prix prédit
    fig = px.scatter(x=pred, y=residus, opacity=0.6,
                     labels={'x': 'Prix prédit (MAD)', 'y': 'Erreur = réel - prédit (MAD)'},
                     title=f'{nom} : erreurs vs prix prédit')
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()

    # 3. distribution des erreurs
    fig = px.histogram(x=residus, nbins=40,
                       labels={'x': 'Erreur (MAD)'},
                       title=f'{nom} : distribution des erreurs')
    fig.show()

graphes_meilleur_modele(pipes[meilleur], meilleur, X_test, y_test)

### Interprétation

#### Comparaison des modèles (R² train / validation / test)
On compare les trois R² de chaque modèle : un R² élevé et proche sur les trois jeux indique un modèle qui généralise bien.

#### Overfitting (surapprentissage)
L'écart `R2_train − R2_test` mesure le surapprentissage : plus il est grand, plus le modèle mémorise le train au lieu de généraliser.

#### Meilleur modèle et ses erreurs (MAE / RMSE)
La MAE donne l'erreur moyenne en MAD ; le RMSE pénalise davantage les grosses erreurs. Les graphiques « prix réel vs prix prédit » et « distribution des erreurs » montrent si les erreurs sont centrées sur 0 et sans biais selon le niveau de prix.

---
### Résumé de la partie 5 : Modélisation

- **10 modèles de régression** comparés : Linear Regression, Ridge, Lasso, KNN, SVR, Decision Tree, Random Forest, Gradient Boosting, XGBoost et CatBoost.
- **Évaluation automatisée** : chaque modèle est entraîné dans un pipeline (prétraitement + modèle) et évalué par le R² (train, validation, test), l'écart d'overfitting, et la MAE et le RMSE en MAD (prix réel).
- **Sélection** du meilleur modèle selon le R² de validation ; le R² de test donne le score final.
- **Sauvegarde** du pipeline complet du meilleur modèle dans `meilleur_modele.pkl`.
- **Visualisations** : comparaison des modèles (R², overfitting, MAE, RMSE), puis, pour le meilleur modèle, prix réel vs prédit, erreurs vs prix prédit et distribution des erreurs.